# Breakout Room #4 — La curva Qini: ¿el modelo ordena de verdad?

**Objetivo:** construir la curva Qini y la tabla de deciles para comparar dos formas de
ordenar a los 14,000 clientes del experimento — por **riesgo/churn** y por **uplift
(T-learner)** — y ver cuál de las dos sirve de verdad para decidir a quién contactar.

Esta vez separamos los datos en dos partes: **entrenamiento** (para que los modelos
aprendan) y **prueba** (para evaluarlos con datos que nunca vieron — la misma idea de "las
dos ventanas" del Breakout 2, pero aplicada a modelos en vez de a fechas).

Donde dice **`# TODO`** hay algo para que ustedes completen.


In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("/Volumes/workspace/default/semana3/uplift_experiment.csv")
FEATURES = ["recency_days", "frequency", "monetary", "email_open_rate"]

train, test = train_test_split(df, test_size=0.30, random_state=42, stratify=df["treatment_group"])
print(f"Entrenamiento: {len(train):,} clientes · Prueba: {len(test):,} clientes (el modelo nunca los ve)")


## Paso 1 — entrenen los dos modelos (solo con datos de ENTRENAMIENTO)

- **`modelo_riesgo`**: un modelo de churn normal y corriente. No sabe nada de tratamiento/control
  — mezcla a todos los clientes de entrenamiento y aprende a predecir quién NO va a responder
  (`churned = 1 - responded_90d`). Así funcionaría un modelo de churn tradicional, el mismo tipo
  que construyeron en el Breakout 2.
- **`modelo_tratado`** y **`modelo_control`**: el T-learner de siempre (Breakout 3), entrenado
  solo con las filas de entrenamiento.


In [0]:
# --- SOLUCIÓN: modelo de riesgo/churn (no sabe de tratamiento) ---
train_riesgo = train.copy()
train_riesgo["churned"] = 1 - train_riesgo["responded_90d"]

modelo_riesgo = LogisticRegression(max_iter=1000)
modelo_riesgo.fit(train_riesgo[FEATURES], train_riesgo["churned"])

# T-learner: un modelo por grupo, SOLO con train
train_t = train[train["treatment_group"] == "treatment"]
train_c = train[train["treatment_group"] == "control"]

modelo_tratado = LogisticRegression(max_iter=1000)
modelo_control = LogisticRegression(max_iter=1000)
modelo_tratado.fit(train_t[FEATURES], train_t["responded_90d"])
modelo_control.fit(train_c[FEATURES], train_c["responded_90d"])

print("modelo_riesgo, modelo_tratado y modelo_control entrenados")


## Paso 2 — puntúen el conjunto de PRUEBA (el que ningún modelo vio)

Igual que en el Breakout 3: le preguntan a los modelos sobre todos los clientes de prueba,
tratados y de control por igual.


In [0]:
test = test.copy()
X_test = test[FEATURES]

# --- SOLUCIÓN: puntuamos el conjunto de prueba (que ningún modelo vio) ---
test["prob_churn"] = modelo_riesgo.predict_proba(X_test)[:, 1]
test["p_tratado"] = modelo_tratado.predict_proba(X_test)[:, 1]
test["p_control"] = modelo_control.predict_proba(X_test)[:, 1]
test["uplift_estimado"] = test["p_tratado"] - test["p_control"]

test[["prob_churn", "p_tratado", "p_control", "uplift_estimado"]].describe().round(3)


## Paso 3 — la curva Qini

Para cada corte N (del 2.5% al 100% de la base de prueba, ordenada de "mejor" a "peor" según el
modelo), midan:

```
ganancia(N) = (tasa de respuesta de los tratados en el top N  −  tasa de respuesta del control en el top N)  ×  N
```

Esa es la curva — compara contra la diagonal del azar. El **coeficiente Qini** es, sencillamente,
el área entre la curva del modelo y la diagonal del azar: entre más área (más arriba de la
diagonal), mejor ordena el modelo.


In [0]:
def curva_qini(test_df, score_col, steps=40):
    orden = test_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    es_tratado = (orden["treatment_group"] == "treatment").values
    respondio = orden["responded_90d"].values

    acumulado_tratados = np.cumsum(np.where(es_tratado, respondio, 0))
    acumulado_control = np.cumsum(np.where(~es_tratado, respondio, 0))
    n_tratados_acum = np.cumsum(es_tratado)
    n_control_acum = np.cumsum(~es_tratado)

    n = len(orden)
    xs, ys = [0.0], [0.0]
    for i in np.linspace(1, n, steps).astype(int):
        nt, nc = n_tratados_acum[i - 1], n_control_acum[i - 1]
        tasa_t = acumulado_tratados[i - 1] / nt if nt > 0 else 0
        tasa_c = acumulado_control[i - 1] / nc if nc > 0 else 0
        xs.append(i / n)
        ys.append((tasa_t - tasa_c) * n)
    return np.array(xs), np.array(ys)


try:
    _trap = np.trapezoid
except AttributeError:
    _trap = np.trapz


def coeficiente_qini(xs, ys):
    area_modelo = _trap(ys, xs)
    area_azar = _trap(np.linspace(0, ys[-1], len(xs)), xs)
    return (area_modelo - area_azar) / len(test)


xs_uplift, ys_uplift = curva_qini(test, "uplift_estimado")
xs_riesgo, ys_riesgo = curva_qini(test, "prob_churn")

qini_uplift = coeficiente_qini(xs_uplift, ys_uplift)
qini_riesgo = coeficiente_qini(xs_riesgo, ys_riesgo)

print(f"Coeficiente Qini · modelo de uplift (T-learner): {qini_uplift:+.3f}")
print(f"Coeficiente Qini · modelo de riesgo (churn):      {qini_riesgo:+.3f}")


In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(xs_uplift, ys_uplift, color="#00704A", linewidth=3, label=f"Modelo de uplift (Qini={qini_uplift:+.3f})")
ax.plot(xs_riesgo, ys_riesgo, color="#C4512F", linewidth=3, label=f"Modelo de riesgo/churn (Qini={qini_riesgo:+.3f})")
ax.plot([0, 1], [0, ys_uplift[-1]], color="#9A988E", linestyle="--", label="Al azar")
ax.set_xlabel("% de la base contactada (de prueba)")
ax.set_ylabel("Respuestas incrementales acumuladas")
ax.set_title("La curva Qini del experimento")
ax.legend()
plt.tight_layout()
plt.savefig("/Volumes/workspace/default/semana3/curva_qini.png", dpi=150)
plt.show()


## Paso 4 — la tabla de deciles: ¿el modelo ordena de verdad?

Dividan el conjunto de prueba en 10 deciles según `uplift_estimado` (decil 1 = uplift más alto
predicho). Para cada decil, comparen el uplift PREDICHO promedio contra el uplift REAL (tasa de
respuesta de tratados menos tasa de respuesta de control, DENTRO de ese decil).


In [0]:
test["decil"] = pd.qcut(test["uplift_estimado"], 10, labels=False, duplicates="drop")
test["decil"] = 9 - test["decil"]  # decil 1 = el de mayor uplift predicho

filas = []
for d in range(10):
    grp = test[test["decil"] == d]
    pred = grp["uplift_estimado"].mean()
    real_t = grp.loc[grp["treatment_group"] == "treatment", "responded_90d"].mean()
    real_c = grp.loc[grp["treatment_group"] == "control", "responded_90d"].mean()
    filas.append({"decil": d + 1, "uplift_predicho": round(pred, 3), "uplift_real": round(real_t - real_c, 3), "n": len(grp)})

tabla_deciles = pd.DataFrame(filas)
tabla_deciles


## Para la Clase

**1. ¿El coeficiente Qini del modelo de riesgo/churn es mayor, menor o parecido al del modelo de uplift? ¿Qué dice eso sobre usar un modelo de churn para decidir a quién contactar?**

   Mucho menor: uplift = **+0.222** vs riesgo/churn = **+0.006** (casi igual que el azar). Un modelo de churn **NO sirve** para decidir a quién contactar: marca a quien se va (Lost Cause), no a quien responde al incentivo. Para targeting hay que usar **uplift**.

**2. Miren la tabla de deciles: ¿hay algún decil donde lo predicho y lo real se alejan mucho? Con ~420 clientes por decil, ¿qué tan confiables son esos números?**

   El **decil 6** es el que más se aleja (predijo +0.099, real -0.059, diferencia 0.158); el **decil 1** también (0.394 vs 0.505). Con ~420 por decil (~210 tratados y ~210 control) el error estándar es ≈ **±0.05**, así que los deciles del medio son ruidosos. Confiable: la **tendencia general** (top claramente positivo, cola cerca de cero); NO confiable: el número exacto de un decil aislado.

**3. Si tuvieran que explicarle a alguien de negocio qué es el coeficiente Qini en una frase, ¿qué dirían?**

   Es cuánta **ganancia extra sobre elegir al azar** consigues al contactar a los primeros de tu lista: 0 = tu orden no sirve más que el azar; más alto = mejor concentras a los clientes que el cupón de verdad convence.
